# Building and Running Models

This notebook shows how to create neural network layers, compose them into
a model, run a forward pass, and train.

## Layer Types

idris-ml has a rich layer system. Let's explore what's available.

In [1]:
:t linearLayer

Layer.Linear.linearLayer : (Num ty, FromDouble ty) => IO (AnyLayer i o ty)


In [2]:
:t rnnLayer

Layer.Rnn.rnnLayer : (Num ty, FromDouble ty) => IO (AnyLayer i o ty)


In [3]:
:t lstmLayer

Layer.Lstm.lstmLayer : (Num ty, FromDouble ty) => IO (AnyLayer i o ty)


In [4]:
:t softmaxLayer

Layer.Normalization.softmaxLayer : (Fractional ty, Floating ty) =>
AnyLayer n n ty


## Creating a Model

Layers are composed with `~>` and terminated with `OutputLayer`.
`autoName` assigns parameter names for gradient tracking.

In [5]:
:exec do { ll <- linearLayer {i=2, o=3};
  model <- pure (autoName (ll ~> OutputLayer softmaxLayer));
  putStrLn (show model) }

Linear<2:3> ~> Normalization<softmax>


## Forward Pass

Create an input tensor and run it through the model.
`forwardVarTensor` returns `(updatedModel, outputTensor)`.

In [6]:
:exec do { ll <- linearLayer {i=2, o=3};
  model <- pure (autoName (OutputLayer ll));
  buf <- pure (prim__setDouble (prim__setDouble (prim__allocDoubles 2) 0 1.0) 1 2.0);
  inT <- pure (prim__createState1d 2 buf);
  pair <- pure (forwardVarTensor model inT);
  outT <- pure (snd pair);
  putStrLn ("output[0] = " ++ show (prim__item1d outT 0));
  putStrLn ("output[1] = " ++ show (prim__item1d outT 1));
  putStrLn ("output[2] = " ++ show (prim__item1d outT 2));
  putStrLn ("sum = " ++ show (prim__item (prim__sum outT))) }

output[0] = -1.0105921685961734
output[1] = -2.4643827449632236
output[2] = -1.3582954020902138
sum = -4.833270315649611


This is the raw linear output (no softmax). Add `softmaxLayer` to get a probability distribution that sums to 1.

## Deeper Networks

Chain multiple layers with `~>`. Type-level dimension checking ensures
adjacent layers are compatible.

In [7]:
:exec do { l1 <- linearLayer {i=4, o=8};
  l2 <- linearLayer {i=8, o=3};
  model <- pure (autoName (l1 ~> reluLayer ~> l2 ~> OutputLayer softmaxLayer));
  putStrLn (show model) }

Linear<4:8> ~> Activation<relu> ~> Linear<8:3> ~> Normalization<softmax>


## Type Safety

Mismatched dimensions are caught at compile time. This won't type-check:

In [8]:
:exec do { l1 <- linearLayer {i=4, o=8};
  l2 <- linearLayer {i=5, o=3};
  model <- pure (autoName (l1 ~> l2 ~> OutputLayer softmaxLayer));
  putStrLn "should not reach here" }

Error: When unifying:
    Network 5 [3] 3 ?ty
and:
    Network 8 [3] 3 ?ty
Mismatch between: 0 and 3.

(Interactive):1:103--1:133
 1 | :exec do { l1 <- linearLayer {i=4, o=8}; l2 <- linearLayer {i=5, o=3}; model <- pure (autoName (l1 ~> l2 ~> OutputLayer softmaxLayer)); putStrLn "should not reach here" }
                                                                                                           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^


The error shows a type mismatch between output dimension 8 and input dimension 5.
This is the key advantage of idris-ml: shape errors are caught before any code runs.

## Optimizers

idris-ml provides SGD, Adam, AdamW, and RMSprop. These are C-level native optimizers.

In [9]:
:t nativeSgd

Variable.nativeSgd : Double -> NativeOptimizer


In [10]:
:t nativeAdamW

Variable.nativeAdamW : Double -> Double -> Double -> Double -> Double -> Double -> NativeOptimizer


## Training

For full training loops, the compiled examples are more practical.
Run them from the terminal:

```bash
make example-supervised   # Linear classification (simplest)
make example-rnn          # RNN pattern prediction
make example-lstm         # LSTM sequence learning
make example-mnist        # CNN on MNIST
make example-gpt          # Character-level language model
make example-reinforce    # REINFORCE on CartPole
```

All examples accept `--epochs`, `--lr`, and `--seed` flags.

## Exploring the API

Use `:browse` to discover what's available in each module.

In [11]:
:browse Optimizer

MkOptimizer : (SortedMap String Double -> OptimizerState -> (SortedMap String Double,
              OptimizerState)) -> Optimizer
MkOptimizerState : SortedMap String Double -> SortedMap String Double -> Int -> OptimizerState
Optimizer : Type
OptimizerState : Type
adam : Double -> Double -> Double -> Double -> Double -> Optimizer
adamGlobalClip : Double -> Double -> Double -> Double -> Double -> Optimizer
applyDeltas : SortedMap String Double -> Variable -> Variable
clipGlobalNorm : Double -> SortedMap String Double -> SortedMap String Double
clipGradValue : Double -> SortedMap String Double -> SortedMap String Double
initState : OptimizerState
m : OptimizerState -> SortedMap String Double
rmsprop : Double -> Double -> Double -> Optimizer
rmspropValueClip : Double -> Double -> Double -> Double -> Optimizer
sgd : Double -> Double -> Optimizer
step : Optimizer -> SortedMap String Double -> OptimizerState -> (SortedMap String Double,
       OptimizerState)
t : OptimizerState -> Int
v : Opt

In [12]:
:browse Layer.Conv

AvgPool1DState : Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Type -> Type
AvgPool2DState : Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Type -> Type
Conv1DState : Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Type -> Type
Conv2DState : Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Type -> Type
ConvOutDim : Nat -> Nat -> Nat -> Nat
MaxPool1DState : Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Type -> Type
MaxPool2DState : Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Nat -> Type -> Type
MkAvgPool1D : (0 _ : inputSize = c * len) ->
              (0 _ : outputSize = c * PoolOutDim len poolK str) ->
              AvgPool1DState c len poolK str inputSize outputSize ty
MkAvgPool2D : (0 _ : inputSize = c * (inH * inW)) ->
              (0 _ : outputSize = c * (PoolOutDim inH poolH strH * PoolOutDim inW poolW strW)) ->
              AvgPool2DState c inH inW poolH poolW strH strW inputSize outputSize ty
MkConv1D : (0 _ : inputSize = inC * len) -

In [13]:
:browse Schedule

Schedule : Type
constant : Double -> Schedule
cosineAnnealing : Double -> Double -> Nat -> Schedule
cosineWithWarmup : Double -> Double -> Nat -> Nat -> Schedule
oneCycle : Double -> Double -> Double -> Double -> Nat -> Schedule
withWarmup : Nat -> Double -> Schedule -> Schedule
